<a href="https://colab.research.google.com/github/AdAdalan/NLP-Assignment3/blob/main/notebooks/alan_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Setting up


In [ ]:
import os

%cd /content

if not os.path.exists("/content/NLP-Assignment3"):
    !git clone https://github.com/AdAdalan/NLP-Assignment3.git
else:
    print("Repo already exists.")

%cd /content/NLP-Assignment3
!ls


# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

/content
Cloning into 'NLP-Assignment3'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 31 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 11.46 KiB | 2.86 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/NLP-Assignment3
notebooks  outputs  src
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Load Data

In [ ]:
# Load data
import json

DATA_DIR = "/content/drive/MyDrive/Colab_Notebooks/NLP_Assignment3/COMP90042_2026-main/data"

with open(f"{DATA_DIR}/train-claims.json", "r") as f:
    train_claims = json.load(f)

with open(f"{DATA_DIR}/dev-claims.json", "r") as f:
    dev_claims = json.load(f)

with open(f"{DATA_DIR}/test-claims-unlabelled.json", "r") as f:
    test_claims = json.load(f)

with open(f"{DATA_DIR}/evidence.json", "r") as f:
    evidence = json.load(f)

print("Train claims:", len(train_claims))
print("Dev claims:", len(dev_claims))
print("Test claims:", len(test_claims))
print("Evidence passages:", len(evidence))

Train claims: 1228
Dev claims: 154
Test claims: 153
Evidence passages: 1208827


In [ ]:
from collections import Counter

train_labels = [item["claim_label"] for item in train_claims.values()]
dev_labels = [item["claim_label"] for item in dev_claims.values()]

print("Train label distribution:")
print(Counter(train_labels))

print("\nDev label distribution:")
print(Counter(dev_labels))

Train label distribution:
Counter({'SUPPORTS': 519, 'NOT_ENOUGH_INFO': 386, 'REFUTES': 199, 'DISPUTED': 124})

Dev label distribution:
Counter({'SUPPORTS': 68, 'NOT_ENOUGH_INFO': 41, 'REFUTES': 27, 'DISPUTED': 18})


In [ ]:
labels = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]

for target_label in labels:
    print("=" * 80)
    print("Label:", target_label)

    for claim_id, item in train_claims.items():
        if item["claim_label"] == target_label:
            print("Claim ID:", claim_id)
            print("Claim text:", item["claim_text"])
            print("Evidence IDs:", item["evidences"])

            print("\nEvidence texts:")
            for eid in item["evidences"]:
                print("-", evidence[eid])
            break

Label: SUPPORTS
Claim ID: claim-2510
Claim text: In 1946, PDO switched to a cool phase.
Evidence IDs: ['evidence-530063', 'evidence-984887']

Evidence texts:
- There is evidence of reversals in the prevailing polarity (meaning changes in cool surface waters versus warm surface waters within the region) of the oscillation occurring around 1925, 1947, and 1977; the last two reversals corresponded with dramatic shifts in salmon production regimes in the North Pacific Ocean.
- 1945/1946: The PDO changed to a "cool" phase, the pattern of this regime shift is similar to the 1970s episode with maximum amplitude in the subarctic and subtropical front but with a greater signature near the Japan while the 1970s shift was stronger near the American west coast.
Label: REFUTES
Claim ID: claim-126
Claim text: El Niño drove record highs in global temperatures suggesting rise may not be down to man-made emissions.
Evidence IDs: ['evidence-338219', 'evidence-1127398']

Evidence texts:
- While ‘climate 

In [ ]:
evidence_ids = list(evidence.keys())
evidence_texts = list(evidence.values())

print("Number of evidence passages:", len(evidence_ids))
print("First evidence ID:", evidence_ids[0])
print("First evidence text:", evidence_texts[0])

Number of evidence passages: 1208827
First evidence ID: evidence-0
First evidence text: John Bennet Lawes, English entrepreneur and agricultural scientist


# 2.TF IDF Retrival + Majority Lableling baseline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000
)

evidence_tfidf = tfidf_vectorizer.fit_transform(evidence_texts)

print("Evidence TF-IDF matrix shape:", evidence_tfidf.shape)

Evidence TF-IDF matrix shape: (1208827, 50000)


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_top_k(claim_text, k=5):
    claim_tfidf = tfidf_vectorizer.transform([claim_text])
    scores = cosine_similarity(claim_tfidf, evidence_tfidf).flatten()

    top_indices = np.argsort(scores)[::-1][:k]
    top_evidence_ids = [evidence_ids[i] for i in top_indices]

    return top_evidence_ids

In [ ]:
claim_id = list(train_claims.keys())[0]
claim_text = train_claims[claim_id]["claim_text"]

retrieved_eids = retrieve_top_k(claim_text, k=5)

print("Claim ID:", claim_id)
print("Claim text:", claim_text)

print("\nRetrieved evidence:")
for eid in retrieved_eids:
    print(eid, ":", evidence[eid])

print("\nGold evidence:")
for eid in train_claims[claim_id]["evidences"]:
    print(eid, ":", evidence[eid])

Claim ID: claim-1937
Claim text: Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.

Retrieved evidence:
evidence-668884 : CO2).
evidence-1005071 : 4-oxalocrotonate 2-oxopent-4-enoate + CO2
evidence-66273 : Higher atmospheric CO2 concentrations have led to an increase in dissolved CO2, which causes ocean acidification.
evidence-893504 : GLOBALVIEW-CO2 is one of these products.
evidence-743151 : All while using and neutralizing CO2 emissions.

Gold evidence:
evidence-442946 : At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse.
evidence-1194317 : Plants can grow as much as 50 percent faster in concentrations of 1,000 ppm CO 2 when compared with ambient conditions, though

In [ ]:
from collections import Counter

train_label_counts = Counter(
    item["claim_label"] for item in train_claims.values()
)

print(train_label_counts)
print("Most common label:", train_label_counts.most_common(1)[0])

Counter({'SUPPORTS': 519, 'NOT_ENOUGH_INFO': 386, 'REFUTES': 199, 'DISPUTED': 124})
Most common label: ('SUPPORTS', 519)


In [ ]:
dev_predictions = {}

for claim_id, item in dev_claims.items():
    claim_text = item["claim_text"]
    retrieved_eids = retrieve_top_k(claim_text, k=5)

    dev_predictions[claim_id] = {
        "claim_label": "SUPPORTS",
        "evidences": retrieved_eids
    }

print("Number of predictions:", len(dev_predictions))

first_id = list(dev_predictions.keys())[0]
print(first_id)
print(dev_predictions[first_id])

Number of predictions: 154
claim-752
{'claim_label': 'SUPPORTS', 'evidences': ['evidence-509345', 'evidence-509525', 'evidence-252686', 'evidence-580844', 'evidence-589336']}


In [ ]:
import json

DEV_PRED_PATH = "/content/drive/MyDrive/Colab_Notebooks/NLP_Assignment3/outputs/dev_predictions_tfidf.json"

with open(DEV_PRED_PATH, "w") as f:
    json.dump(dev_predictions, f, indent=2)

print("Saved to:", DEV_PRED_PATH)

Saved to: /content/drive/MyDrive/Colab_Notebooks/NLP_Assignment3/outputs/dev_predictions_tfidf.json


In [ ]:
!python /content/drive/MyDrive/Colab_Notebooks/NLP_Assignment3/COMP90042_2026-main/eval.py \
  --predictions /content/drive/MyDrive/Colab_Notebooks/NLP_Assignment3/outputs/dev_predictions_tfidf.json\
  --groundtruth "/content/drive/MyDrive/Colab_Notebooks/NLP_Assignment3/COMP90042_2026-main/data/dev-claims.json"

Evidence Retrieval F-score (F)    = 0.08808493094207381
Claim Classification Accuracy (A) = 0.44155844155844154
Harmonic Mean of F and A          = 0.1468710715587297


# 3. TF IDF TOP K + Sentence Transformer Ranking

In [ ]:
def evaluate_retrieval_recall_at_k(claims, k_values=[5, 10, 20, 50, 100, 200]):
    results = {}

    for k in k_values:
        total_claims = 0
        any_hit_count = 0
        full_hit_count = 0
        total_gold_evidence = 0
        retrieved_gold_evidence = 0

        for claim_id, item in claims.items():
            claim_text = item["claim_text"]
            gold_eids = set(item["evidences"])

            retrieved_eids = set(retrieve_top_k(claim_text, k=k))

            total_claims += 1

            # 至少命中一个 gold evidence
            if len(gold_eids & retrieved_eids) > 0:
                any_hit_count += 1

            # 是否命中全部 gold evidence
            if gold_eids.issubset(retrieved_eids):
                full_hit_count += 1

            # evidence-level recall
            total_gold_evidence += len(gold_eids)
            retrieved_gold_evidence += len(gold_eids & retrieved_eids)

        results[k] = {
            "any_hit_rate": any_hit_count / total_claims,
            "full_hit_rate": full_hit_count / total_claims,
            "evidence_recall": retrieved_gold_evidence / total_gold_evidence
        }

    return results

In [ ]:
recall_results = evaluate_retrieval_recall_at_k(
    dev_claims,
    k_values=[5, 10, 20, 50, 100, 200, 500]
)

for k, scores in recall_results.items():
    print(f"K = {k}")
    print(f"  Any-hit rate:       {scores['any_hit_rate']:.4f}")
    print(f"  Full-hit rate:      {scores['full_hit_rate']:.4f}")
    print(f"  Evidence recall:    {scores['evidence_recall']:.4f}")
    print()

K = 5
  Any-hit rate:       0.3052
  Full-hit rate:      0.0455
  Evidence recall:    0.1120

K = 10
  Any-hit rate:       0.3896
  Full-hit rate:      0.0649
  Evidence recall:    0.1507

K = 20
  Any-hit rate:       0.4610
  Full-hit rate:      0.0779
  Evidence recall:    0.1914

K = 50
  Any-hit rate:       0.5714
  Full-hit rate:      0.1364
  Evidence recall:    0.2953

K = 100
  Any-hit rate:       0.6623
  Full-hit rate:      0.1753
  Evidence recall:    0.3646

K = 200
  Any-hit rate:       0.7662
  Full-hit rate:      0.2338
  Evidence recall:    0.4501

K = 500
  Any-hit rate:       0.8506
  Full-hit rate:      0.3247
  Evidence recall:    0.5621



# CPU 版本

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path("/content/drive/MyDrive/Colab_Notebooks/NLP_Assignment3/COMP90042_2026-main/data")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import json

#read data
with open(f"{DATA_DIR}/train-claims.json", "r") as f:
    train_claims = json.load(f)

with open(f"{DATA_DIR}/dev-claims.json", "r") as f:
    dev_claims = json.load(f)

with open(f"{DATA_DIR}/test-claims-unlabelled.json", "r") as f:
    test_claims = json.load(f)

with open(f"{DATA_DIR}/evidence.json", "r") as f:
    evidence = json.load(f)

print("Train claims length:", len(train_claims))
print("Dev claims length:", len(dev_claims))
print("Test claims length:", len(test_claims))
print("Evidence passages length:", len(evidence))

Train claims length: 1228
Dev claims length: 154
Test claims length: 153
Evidence passages length: 1208827


In [10]:
from collections import Counter
import numpy as np

# 统计训练集标签分布
label_dist = Counter(c["claim_label"] for c in train_claims.values())

print("Training Set Label Distribution:")
for lbl, cnt in label_dist.most_common():
    print(f"{lbl:20s} {cnt:5d} ({cnt / len(train_claims):.1%})")

# 统计每条 claim 对应的 GT evidence 数量
gt_counts = [len(c["evidences"]) for c in train_claims.values()]

print(
    f"\nGround Truth Evidence For Each Claim: "
    f"min={min(gt_counts)}, "
    f"max={max(gt_counts)}, "
    f"mean={np.mean(gt_counts):.2f}, "
    f"median={int(np.median(gt_counts))}"
)

Training Set Label Distribution:
SUPPORTS               519 (42.3%)
NOT_ENOUGH_INFO        386 (31.4%)
REFUTES                199 (16.2%)
DISPUTED               124 (10.1%)

Ground Truth Evidence For Each Claim: min=1, max=5, mean=3.36, median=3


In [11]:
import random

sample_evidence = random.sample(list(evidence.items()), 20)

for eid, text in sample_evidence:
    print("ID:", eid)
    print(text)
    print("-" * 80)

ID: evidence-410826
Has also worked in the Defence Logistics Organisation and the Defence Ministry.
--------------------------------------------------------------------------------
ID: evidence-106446
Colden is an interior town in the south part of the county.
--------------------------------------------------------------------------------
ID: evidence-708080
In the second paradigm, the basic management module is soldered to the motherboard and the OPMA connector is used as an upgrade path for advanced platform management features.
--------------------------------------------------------------------------------
ID: evidence-143995
Some of the rockets had blades in the front of the bamboo guiding rods, while others were designed as incendiary rockets.
--------------------------------------------------------------------------------
ID: evidence-581858
It also bans smoking in outdoor recreational or educational areas such as parks, stadiums and university campuses.
-----------------------

In [12]:
sample_claims = random.sample(list(train_claims.items()), 20)

for cid, c in sample_claims:
    print("ID:", cid)
    print(c["claim_text"])
    print("-" * 80)

ID: claim-19
If CO2 was so terrible for the planet, then installing a CO2 generator in a greenhouse would kill the plants.
--------------------------------------------------------------------------------
ID: claim-2189
This makes it clear that this time around humans are the cause, mainly by our CO2 emissions.
--------------------------------------------------------------------------------
ID: claim-663
“For example, Canadian polar bear biologist Ian Stirling learned in the 1970s that spring sea ice in the southern Beaufort Sea periodically gets so thick that seals depart, depriving local polar bears of their prey and causing their numbers to plummet.
--------------------------------------------------------------------------------
ID: claim-1627
In the last 35 years of global warming, sun and climate have been going in opposite directions.
--------------------------------------------------------------------------------
ID: claim-2656
The divergence of tree-ring proxies from temperature

## Retrival TFI IDF Baseline

In [13]:
# Baseline TFIDF

import nltk
nltk.download("stopwords")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from nltk.corpus import stopwords


def tfidf(evidences, claims):
    vectorizer = TfidfVectorizer(
        stop_words=stopwords.words("english")
    )

    evidence_tfidf = vectorizer.fit_transform(evidences)
    claims_tfidf = vectorizer.transform(claims)

    cos_similarity = cosine_similarity(claims_tfidf, evidence_tfidf)

    return cos_similarity

def choose_top_k(cos_similarity, evidence_ids, claim_ids, top_k):
    predict_tfidf = {}

    for i, cid in enumerate(claim_ids):
        cos_row = cos_similarity[i]

        # 找出当前 claim 相似度最高的 top_k 个 evidence 的 index
        top_idx = np.argpartition(-cos_row, top_k)[:top_k]  # O(n)

        # 根据 index 找回 evidence_id
        predict_tfidf[cid] = [evidence_ids[idx] for idx in top_idx]

    return predict_tfidf

def eval_retrieval(claims_dataset, predict):
    all_recalls = []
    all_precisions = []
    all_fscores = []

    for claim_id, claim in sorted(claims_dataset.items()):

        if claim_id not in predict:
            continue

        evidence_correct = 0
        evidence_recall = 0.0
        evidence_precision = 0.0
        evidence_fscore = 0.0

        if isinstance(predict[claim_id], list) and len(predict[claim_id]) > 0:
            predict_set = set(predict[claim_id])

            for true_id in claim["evidences"]:
                if true_id in predict_set:
                    evidence_correct += 1

            if evidence_correct > 0:
                evidence_recall = evidence_correct / len(claim["evidences"])
                evidence_precision = evidence_correct / len(predict[claim_id])
                evidence_fscore = (
                    2 * evidence_precision * evidence_recall
                ) / (
                    evidence_precision + evidence_recall
                )

        all_recalls.append(evidence_recall)
        all_precisions.append(evidence_precision)
        all_fscores.append(evidence_fscore)

        min_recall =min(all_recalls)


    print(f"Mean Recall:    {np.mean(all_recalls):.6f}")
    print(f"Mean Precision: {np.mean(all_precisions):.6f}")
    print(f"Mean F1-Score:  {np.mean(all_fscores):.6f}")
    print(f"Min Recall:     {min_recall:.6f}")

    return np.mean(all_fscores), np.mean(all_precisions), np.mean(all_recalls), min_recall

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
evidence_list_ids = list(evidence.keys())
evidence_list_texts = list(evidence.values())
dev_list_ids = list(dev_claims.keys())
dev_list_claims = list(dev_claims[id]["claim_text"] for id in dev_list_ids)

print(evidence_list_ids[3])
print(evidence_list_texts[3])
print(dev_list_ids[3])
print(dev_list_claims[3])

evidence-3
Gerald Francis Goyer (born October 20, 1936) was a professional ice hockey player who played 40 games in the National Hockey League.
claim-871
“As it happens, Zika may also be a good model of the second worrying effect — disease mutation.


In [15]:
Top_K = 100000
cos_row = tfidf(evidence_list_texts, dev_list_claims)
predict_tfidf = choose_top_k(cos_row, evidence_list_ids, dev_list_ids, Top_K)
evaluation = eval_retrieval(dev_claims, predict_tfidf)



Mean Recall:    0.945022
Mean Precision: 0.000030
Mean F1-Score:  0.000059
Min Recall:     0.000000


In [16]:
predict_R1 = predict_tfidf

In [17]:
def flatten_evidence_ids(evidences):
    """
    兼容两种格式：
    1. ["evidence-1", "evidence-2"]
    2. [["evidence-1"], ["evidence-2"]]
    """
    flat = []

    for ev in evidences:
        if isinstance(ev, list):
            flat.extend(ev)
        else:
            flat.append(ev)

    return flat


def find_zero_recall_claims(claims_dict, predict_dict):
    """
    找出 retrieval 没有命中任何 GT evidence 的 claim。
    """
    zero_claims = []

    for cid, c in claims_dict.items():
        gt_evs = set(flatten_evidence_ids(c["evidences"]))
        pred_evs = set(predict_dict.get(cid, []))

        hit = gt_evs.intersection(pred_evs)

        if len(hit) == 0:
            zero_claims.append(cid)

    return zero_claims

In [18]:
zero_cids = find_zero_recall_claims(dev_claims, predict_R1)

print("Zero-recall claims:", len(zero_cids))
print("Total dev claims:", len(dev_claims))
print("Ratio:", len(zero_cids) / len(dev_claims))

Zero-recall claims: 1
Total dev claims: 154
Ratio: 0.006493506493506494


In [21]:
def inspect_zero_recall_claims(
    zero_cids,
    claims_dict,
    predict_dict,
    evidence_dict,
    n=10,
    show_top_pred=5
):
    """
    打印 zero-recall claims 的 claim text、label、GT evidence、预测 evidence。
    """
    for idx, cid in enumerate(zero_cids[:n], start=1):
        c = claims_dict[cid]

        gt_evs = flatten_evidence_ids(c["evidences"])
        pred_evs = predict_dict.get(cid, [])

        print("=" * 100)
        print(f"[{idx}] Claim ID: {cid}")
        print("Label:", c.get("claim_label", "N/A"))
        print("\nClaim text:")
        print(c["claim_text"])

        print("\nGT evidence IDs:")
        print(gt_evs)

        print("\nGT evidence texts:")
        for eid in gt_evs:
            print(f"- {eid}:")
            print(" ", evidence_dict.get(eid, "[EVIDENCE ID NOT FOUND]"))

        print(f"\nTop-{show_top_pred} predicted evidence IDs:")
        print(pred_evs[:show_top_pred])

        print(f"\nTop-{show_top_pred} predicted evidence texts:")
        for eid in pred_evs[:show_top_pred]:
            print(f"- {eid}:")
            print(" ", evidence_dict.get(eid, "[EVIDENCE ID NOT FOUND]"))

        print()

In [22]:
inspect_zero_recall_claims(
    zero_cids=zero_cids,
    claims_dict=dev_claims,
    predict_dict=predict_R1,
    evidence_dict=evidence,
    n=10,
    show_top_pred=5
)

[1] Claim ID: claim-161
Label: SUPPORTS

Claim text:
Extreme melting and changes to the climate like this has released pressure on to the continent, allowing the ground to rise up.

GT evidence IDs:
['evidence-702708']

GT evidence texts:
- evidence-702708:
  The gravitational effects comes into play when a large ice sheet melts.

Top-5 predicted evidence IDs:
['evidence-1208786', 'evidence-1208645', 'evidence-1208654', 'evidence-1208636', 'evidence-1208601']

Top-5 predicted evidence texts:
- evidence-1208786:
  The Album should be released later in 2013 on the Slice of Spice record label based in Brooklyn NYC.
- evidence-1208645:
  The association held regular meetings in various countries of the continent.
- evidence-1208654:
  The film was released on 3 September 2010 under the Glorious Entertainment banner.
- evidence-1208636:
  The changes would add new features to combat counterfeiting and make them easier for blind citizens to distinguish.
- evidence-1208601:
  The Edict of Wie

## BM25S

In [ ]:
!pip install -q bm25s sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 4.5 MB/s eta 0:00:00


In [ ]:
import time
import bm25s
import psutil

evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]


print("Tokenizing 1.2M evidence ...")
t0 = time.time()

corpus_tokens = bm25s.tokenize(
    evidence_texts,
    stopwords="en",
    stemmer=None
)

print(f"    tokenize 用时: {time.time() - t0:.0f}s")

print("\nbuilding BM25 index ...")
t0 = time.time()

retriever = bm25s.BM25()
retriever.index(corpus_tokens)

print(f"    index 用时: {time.time() - t0:.0f}s")

Tokenizing 1.2M evidence ...


Split strings:   0%|          | 0/1208827 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


    tokenize 用时: 22s

building BM25 index ...


BM25S Count Tokens:   0%|          | 0/1208827 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1208827 [00:00<?, ?it/s]

    index 用时: 47s


In [ ]:
from tqdm.auto import tqdm

def bm25s_retrieve(claims_dict, k=100):
    cids = list(claims_dict.keys())
    texts = [claims_dict[c]["claim_text"] for c in cids]

    query_tokens = bm25s.tokenize(
        texts,
        stopwords="en",
        stemmer=None
    )

    results, scores = retriever.retrieve(
        query_tokens,
        k=k
    )

    out = {}
    for i, cid in enumerate(cids):
        out[cid] = [evidence_ids[j] for j in results[i]]

    return out

In [ ]:
print("BM25s baseline(top-5)")
TOP_K = 100000
t0 = time.time()

predict_R2 = bm25s_retrieve(
    dev_claims,
    k=TOP_K
)

print(f"用时: {time.time() - t0:.1f}s\n")

f_R2 = eval_retrieval(
    dev_claims,
    predict_R2
)


BM25s baseline(top-5)


Split strings:   0%|          | 0/154 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/154 [00:00<?, ?it/s]

用时: 5.2s

Mean Recall:    0.951840
Mean Precision: 0.000030
Mean F1-Score:  0.000060
Min Recall:     0.000000


In [ ]:
def flatten_evidence_ids(evidences):
    """
    兼容两种格式：
    1. ["evidence-1", "evidence-2"]
    2. [["evidence-1"], ["evidence-2"]]
    """
    flat = []

    for ev in evidences:
        if isinstance(ev, list):
            flat.extend(ev)
        else:
            flat.append(ev)

    return flat


def find_zero_recall_claims(claims_dict, predict_dict):
    """
    找出 retrieval 没有命中任何 GT evidence 的 claim。
    """
    zero_claims = []

    for cid, c in claims_dict.items():
        gt_evs = set(flatten_evidence_ids(c["evidences"]))
        pred_evs = set(predict_dict.get(cid, []))

        hit = gt_evs.intersection(pred_evs)

        if len(hit) == 0:
            zero_claims.append(cid)

    return zero_claims

In [ ]:
zero_cids = find_zero_recall_claims(dev_claims, predict_R2)

print("Zero-recall claims:", len(zero_cids))
print("Total dev claims:", len(dev_claims))
print("Ratio:", len(zero_cids) / len(dev_claims))

Zero-recall claims: 1
Total dev claims: 154
Ratio: 0.006493506493506494


In [ ]:
def inspect_zero_recall_claims(
    zero_cids,
    claims_dict,
    predict_dict,
    evidence_dict,
    n=10,
    show_top_pred=5
):
    """
    打印 zero-recall claims 的 claim text、label、GT evidence、预测 evidence。
    """
    for idx, cid in enumerate(zero_cids[:n], start=1):
        c = claims_dict[cid]

        gt_evs = flatten_evidence_ids(c["evidences"])
        pred_evs = predict_dict.get(cid, [])

        print("=" * 100)
        print(f"[{idx}] Claim ID: {cid}")
        print("Label:", c.get("claim_label", "N/A"))
        print("\nClaim text:")
        print(c["claim_text"])

        print("\nGT evidence IDs:")
        print(gt_evs)

        print("\nGT evidence texts:")
        for eid in gt_evs:
            print(f"- {eid}:")
            print(" ", evidence_dict.get(eid, "[EVIDENCE ID NOT FOUND]"))

        print(f"\nTop-{show_top_pred} predicted evidence IDs:")
        print(pred_evs[:show_top_pred])

        print(f"\nTop-{show_top_pred} predicted evidence texts:")
        for eid in pred_evs[:show_top_pred]:
            print(f"- {eid}:")
            print(" ", evidence_dict.get(eid, "[EVIDENCE ID NOT FOUND]"))

        print()

In [ ]:
inspect_zero_recall_claims(
    zero_cids=zero_cids,
    claims_dict=dev_claims,
    predict_dict=predict_R2,
    evidence_dict=evidence,
    n=10,
    show_top_pred=5
)

[1] Claim ID: claim-161
Label: SUPPORTS

Claim text:
Extreme melting and changes to the climate like this has released pressure on to the continent, allowing the ground to rise up.

GT evidence IDs:
['evidence-702708']

GT evidence texts:
- evidence-702708:
  The gravitational effects comes into play when a large ice sheet melts.

Top-5 predicted evidence IDs:
['evidence-169769', 'evidence-750033', 'evidence-1040448', 'evidence-783632', 'evidence-937636']

Top-5 predicted evidence texts:
- evidence-169769:
  This is predicted to produce changes such as the melting of glaciers and ice sheets, more extreme temperature ranges, significant changes in weather and a global rise in average sea levels.
- evidence-750033:
  The Indian sub-continent crumples as it pushes against Asia and pressure is released.
- evidence-1040448:
  Heated ground causes air to rise which results in lower air pressure.
- evidence-783632:
  It is not constant for a given fan, but changes with both air flow rate and 

## pretrained model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Classification

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

LABELS = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


def make_input_text(claim_text, evidence_id_list, evidence_dict, max_evs=5):
    """
    把一条 claim + 它的若干条 evidence 拼成一个字符串，作为分类器的输入。

    用 [SEP] 显式分隔 claim 和 evidence。
    如果用的是 BERT/DeBERTa 这类 transformer，
    tokenizer 自己也会再加特殊 token。
    """
    ev_text = " ".join(
        evidence_dict[eid] for eid in evidence_id_list[:max_evs]
    )

    return claim_text + " [SEP] " + ev_text

Device: cuda


In [ ]:
from collections import Counter


def simple_tokenize(text):
    """
    最朴素的 word 小写 + 按空格切。
    Baseline 不引入 spacy/nltk 复杂分词，保持简单可复现。
    """
    return text.lower().split()


token_counter = Counter()

for cid, c in train_claims.items():
    text = make_input_text(
        c["claim_text"],
        c["evidences"],
        evidence
    )
    token_counter.update(simple_tokenize(text))


VOCAB_MIN_FREQ = 2
VOCAB_MAX_SIZE = 30000

vocab = {
    "<pad>": 0,
    "<unk>": 1
}

for word, cnt in token_counter.most_common(VOCAB_MAX_SIZE):
    if cnt < VOCAB_MIN_FREQ:
        break
    vocab[word] = len(vocab)


def encode(text, max_len=256):
    """
    文本 -> 定长 id 序列。
    短的右侧 pad，长的截断。
    """
    ids = [
        vocab.get(w, 1)
        for w in simple_tokenize(text)
    ][:max_len]

    ids = ids + [0] * (max_len - len(ids))

    return ids

In [ ]:
class ClaimEvDatasetLSTM(Dataset):
    """
    把 (claim, evidence_list, label) 三元组构成 Dataset。

    训练时 retrieval=None，自动用 GT evidence；
    推理/验证时传入 retrieval dict，用检索出的 evidence。
    """

    def __init__(self, claims_dict, evidence_dict, retrieval=None, max_len=256):
        self.cids = list(claims_dict.keys())
        self.claims = claims_dict
        self.evidence = evidence_dict
        self.retrieval = retrieval
        self.max_len = max_len
        self.has_label = "claim_label" in next(iter(claims_dict.values()))

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, i):
        cid = self.cids[i]
        c = self.claims[cid]

        ev_ids = c["evidences"] if self.retrieval is None else self.retrieval[cid]

        text = make_input_text(
            c["claim_text"],
            ev_ids,
            self.evidence
        )

        x = torch.tensor(
            encode(text, self.max_len),
            dtype=torch.long
        )

        if self.has_label:
            y = torch.tensor(
                LABEL2ID[c["claim_label"]],
                dtype=torch.long
            )
            return x, y

        return x, cid


train_ds_lstm = ClaimEvDatasetLSTM(
    train_claims,
    evidence,
    retrieval=None
)

dev_ds_lstm = ClaimEvDatasetLSTM(
    dev_claims,
    evidence,
    retrieval=predict_R1
)

train_loader_lstm = DataLoader(
    train_ds_lstm,
    batch_size=32,
    shuffle=True
)

dev_loader_lstm = DataLoader(
    dev_ds_lstm,
    batch_size=32,
    shuffle=False
)

print(
    f"#train pairs = {len(train_ds_lstm)}, "
    f"#dev pairs = {len(dev_ds_lstm)}"
)

#train pairs = 1228, #dev pairs = 154


In [ ]:
import torch.nn as nn


class BiLSTMClassifier(nn.Module):
    """
    三层结构：
    embedding -> BiLSTM -> masked mean pool -> linear 4类输出

    参数量约 3-4M，T4 上 5 epoch 几分钟跑完。
    """

    def __init__(self, vocab_size, emb_dim=100, hidden=128, n_class=4, dropout=0.3):
        super().__init__()

        self.emb = nn.Embedding(
            vocab_size,
            emb_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            emb_dim,
            hidden,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden * 2,
            n_class
        )

    def forward(self, x):
        # x: [batch_size, seq_len]

        mask = (x != 0).unsqueeze(-1).float()
        # mask: [batch_size, seq_len, 1]
        # 非 padding token 是 1，padding token 是 0

        emb = self.emb(x)
        # emb: [batch_size, seq_len, emb_dim]

        h, _ = self.lstm(emb)
        # h: [batch_size, seq_len, hidden * 2]

        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        # pooled: [batch_size, hidden * 2]
        # 只对非 padding 的位置取平均

        logits = self.fc(self.dropout(pooled))
        # logits: [batch_size, n_class]

        return logits


model_C1 = BiLSTMClassifier(
    vocab_size=len(vocab)
).to(device)

n_params = sum(
    p.numel()
    for p in model_C1.parameters()
)

print(f"BiLSTM 参数量: {n_params:,}")

BiLSTM 参数量: 967,748


In [ ]:
import torch.optim as optim
from collections import defaultdict

label_counts = Counter(c["claim_label"] for c in train_claims.values())

class_weights = torch.tensor(
    [
        len(train_claims) / (len(LABELS) * label_counts[l])
        for l in LABELS
    ],
    dtype=torch.float
).to(device)

print(
    "class weights:",
    dict(zip(LABELS, class_weights.cpu().tolist()))
)

loss_fn = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = optim.Adam(
    model_C1.parameters(),
    lr=1e-3
)

EPOCHS = 3

for epoch in range(EPOCHS):
    model_C1.train()
    epoch_losses = []

    for x, y in train_loader_lstm:
        x, y = x.to(device), y.to(device)

        logits = model_C1(x)

        loss = loss_fn(logits, y)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model_C1.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        epoch_losses.append(loss.item())

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"train_loss = {np.mean(epoch_losses):.4f}"
    )

class weights: {'SUPPORTS': 0.5915221571922302, 'REFUTES': 1.5427135229110718, 'NOT_ENOUGH_INFO': 0.7953367829322815, 'DISPUTED': 2.475806474685669}
Epoch 1/3 train_loss = 1.3698
Epoch 2/3 train_loss = 1.3206
Epoch 3/3 train_loss = 1.2137


In [ ]:
def predict_with_lstm(model, data_loader, dataset):
    """
    把模型拿到每条 claim 的预测标签，
    返回 dict[claim_id -> label_string]
    """
    model.eval()

    all_preds = []

    with torch.no_grad():
        for x, _ in data_loader:
            x = x.to(device)

            logits = model(x)
            preds = logits.argmax(dim=-1).cpu().tolist()

            all_preds.extend(preds)

    return {
        dataset.cids[i]: ID2LABEL[p]
        for i, p in enumerate(all_preds)
    }


pred_label_C1 = predict_with_lstm(
    model_C1,
    dev_loader_lstm,
    dev_ds_lstm
)


correct = sum(
    1
    for cid, plabel in pred_label_C1.items()
    if dev_claims[cid]["claim_label"] == plabel
)

acc_C1 = correct / len(pred_label_C1)

print(f"C1 Classification Accuracy on dev: {acc_C1:.4f}")

C1 Classification Accuracy on dev: 0.2857


# pretrained model

In [ ]:
# Authenticate to Hugging Face
from huggingface_hub import login

login()

In [ ]:
!pip install -q transformers

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm.auto import tqdm
import numpy as np

In [ ]:
LABELS = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_C2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
n_params = sum(
    p.numel()
    for p in model_C2.parameters()
)

print(f"BiLSTM 参数量: {n_params:,}")

BiLSTM 参数量: 66,956,548


In [ ]:
def make_transformer_input(claim_text, evidence_id_list, evidence_dict, max_evs=5):
    ev_texts = [
        evidence_dict[eid]
        for eid in evidence_id_list[:max_evs]
        if eid in evidence_dict
    ]

    evidence_text = " [SEP] ".join(ev_texts)

    return claim_text + " [SEP] " + evidence_text

In [ ]:
class ClaimEvDatasetTransformer(Dataset):
    def __init__(
        self,
        claims_dict,
        evidence_dict,
        retrieval=None,
        tokenizer=None,
        max_len=256,
        max_evs=5
    ):
        self.cids = list(claims_dict.keys())
        self.claims = claims_dict
        self.evidence = evidence_dict
        self.retrieval = retrieval
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.max_evs = max_evs
        self.has_label = "claim_label" in next(iter(claims_dict.values()))

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, i):
        cid = self.cids[i]
        c = self.claims[cid]

        ev_ids = c["evidences"] if self.retrieval is None else self.retrieval[cid]

        text = make_transformer_input(
            c["claim_text"],
            ev_ids,
            self.evidence,
            max_evs=self.max_evs
        )

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "cid": cid
        }

        if self.has_label:
            item["labels"] = torch.tensor(
                LABEL2ID[c["claim_label"]],
                dtype=torch.long
            )

        return item

In [ ]:
train_ds_C2 = ClaimEvDatasetTransformer(
    train_claims,
    evidence,
    retrieval=None,          # train 用 GT evidence
    tokenizer=tokenizer,
    max_len=256,
    max_evs=5
)

dev_ds_C2 = ClaimEvDatasetTransformer(
    dev_claims,
    evidence,
    retrieval=predict_R1,    # dev 用你检索出来的 evidence，比如 BM25 / TF-IDF
    tokenizer=tokenizer,
    max_len=256,
    max_evs=5
)

train_loader_C2 = DataLoader(
    train_ds_C2,
    batch_size=16,
    shuffle=True
)

dev_loader_C2 = DataLoader(
    dev_ds_C2,
    batch_size=16,
    shuffle=False
)

print(
    f"#train = {len(train_ds_C2)}, "
    f"#dev = {len(dev_ds_C2)}"
)

#train = 1228, #dev = 154


In [ ]:
optimizer = AdamW(
    model_C2.parameters(),
    lr=2e-5
)

EPOCHS = 3

for epoch in range(EPOCHS):
    model_C2.train()
    losses = []

    for batch in tqdm(train_loader_C2, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_C2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    print(f"Epoch {epoch+1}/{EPOCHS} train_loss = {np.mean(losses):.4f}")

Epoch 1/3:   0%|          | 0/77 [00:00<?, ?it/s]

Epoch 1/3 train_loss = 1.1749


Epoch 2/3:   0%|          | 0/77 [00:00<?, ?it/s]

Epoch 2/3 train_loss = 0.9010


Epoch 3/3:   0%|          | 0/77 [00:00<?, ?it/s]

Epoch 3/3 train_loss = 0.7373


In [ ]:
def predict_with_transformer(model, data_loader):
    model.eval()

    pred_labels = {}

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            cids = batch["cid"]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = outputs.logits.argmax(dim=-1).cpu().tolist()

            for cid, pred_id in zip(cids, preds):
                pred_labels[cid] = ID2LABEL[pred_id]

    return pred_labels

In [ ]:
pred_label_C2 = predict_with_transformer(
    model_C2,
    dev_loader_C2
)

correct = sum(
    1
    for cid, pred in pred_label_C2.items()
    if dev_claims[cid]["claim_label"] == pred
)

acc_C2 = correct / len(pred_label_C2)

print(f"C2 DistilBERT Classification Accuracy on dev: {acc_C2:.4f}")

Predicting:   0%|          | 0/10 [00:00<?, ?it/s]

C2 DistilBERT Classification Accuracy on dev: 0.3766


### RoBERTa

In [ ]:
MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_C3 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
train_ds_C3 = ClaimEvDatasetTransformer(
    train_claims,
    evidence,
    retrieval=None,          # train 用 GT evidence
    tokenizer=tokenizer,
    max_len=256,
    max_evs=5
)

dev_ds_C3 = ClaimEvDatasetTransformer(
    dev_claims,
    evidence,
    retrieval=predict_R1,    # dev 用检索出来的 evidence
    tokenizer=tokenizer,
    max_len=256,
    max_evs=5
)

train_loader_C3 = DataLoader(
    train_ds_C3,
    batch_size=8,            # RoBERTa 比 DistilBERT 大，先用 8 更稳
    shuffle=True
)

dev_loader_C3 = DataLoader(
    dev_ds_C3,
    batch_size=8,
    shuffle=False
)

print(
    f"#train = {len(train_ds_C3)}, "
    f"#dev = {len(dev_ds_C3)}"
)

#train = 1228, #dev = 154


In [ ]:
optimizer = AdamW(
    model_C3.parameters(),
    lr=2e-5
)

EPOCHS = 3

for epoch in range(EPOCHS):
    model_C3.train()
    losses = []

    for batch in tqdm(train_loader_C3, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_C3(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    print(f"Epoch {epoch+1}/{EPOCHS} train_loss = {np.mean(losses):.4f}")

Epoch 1/3:   0%|          | 0/154 [00:00<?, ?it/s]

Epoch 1/3 train_loss = 1.1732


Epoch 2/3:   0%|          | 0/154 [00:00<?, ?it/s]

Epoch 2/3 train_loss = 0.9385


Epoch 3/3:   0%|          | 0/154 [00:00<?, ?it/s]

Epoch 3/3 train_loss = 0.8202


In [ ]:
def predict_with_transformer(model, data_loader):
    model.eval()

    pred_labels = {}

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            cids = batch["cid"]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = outputs.logits.argmax(dim=-1).cpu().tolist()

            for cid, pred_id in zip(cids, preds):
                pred_labels[cid] = ID2LABEL[pred_id]

    return pred_labels

In [ ]:
pred_label_C3 = predict_with_transformer(
    model_C3,
    dev_loader_C3
)

correct = sum(
    1
    for cid, pred in pred_label_C3.items()
    if dev_claims[cid]["claim_label"] == pred
)

acc_C3 = correct / len(pred_label_C3)

print(f"C3 RoBERTa-base Classification Accuracy on dev: {acc_C3:.4f}")

Predicting:   0%|          | 0/20 [00:00<?, ?it/s]

C3 RoBERTa-base Classification Accuracy on dev: 0.3831


In [ ]:
import gc
import torch

# 删除不用的模型和变量
del model_C1
del model_C2
del model_C3

# 如果还有 optimizer 也可以删
# del optimizer

gc.collect()
torch.cuda.empty_cache()

NameError: name 'model_C1' is not defined

In [ ]:
import torch

print("allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("reserved: ", torch.cuda.memory_reserved() / 1024**3, "GB")

allocated: 1.9001212120056152 GB
reserved:  3.37890625 GB


In [ ]:
import gc
import torch

for var in [
    "model_C1", "model_C2", "model_C3", "model_C4"
    "optimizer",
    "outputs", "loss", "logits",
    "input_ids", "attention_mask", "labels",
    "batch"
]:
    if var in globals():
        del globals()[var]

gc.collect()
torch.cuda.empty_cache()

print("allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("reserved: ", torch.cuda.memory_reserved() / 1024**3, "GB")

allocated: 0.8517284393310547 GB
reserved:  1.111328125 GB


### BERT

In [ ]:
MODEL_NAME = "bert-base-uncased"

In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer_C4 = AutoTokenizer.from_pretrained(MODEL_NAME)

model_C4 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def make_transformer_pair_input(claim_text, evidence_id_list, evidence_dict, max_evs=5):
    ev_texts = [
        evidence_dict[eid]
        for eid in evidence_id_list[:max_evs]
        if eid in evidence_dict
    ]

    evidence_text = " ".join(ev_texts)

    return claim_text, evidence_text

In [ ]:
class ClaimEvDatasetTransformer(Dataset):
    def __init__(
        self,
        claims_dict,
        evidence_dict,
        retrieval=None,
        tokenizer=None,
        max_len=256,
        max_evs=5
    ):
        self.cids = list(claims_dict.keys())
        self.claims = claims_dict
        self.evidence = evidence_dict
        self.retrieval = retrieval
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.max_evs = max_evs
        self.has_label = "claim_label" in next(iter(claims_dict.values()))

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, i):
        cid = self.cids[i]
        c = self.claims[cid]

        ev_ids = c["evidences"] if self.retrieval is None else self.retrieval[cid]

        claim_text, evidence_text = make_transformer_pair_input(
            c["claim_text"],
            ev_ids,
            self.evidence,
            max_evs=self.max_evs
        )

        encoded = self.tokenizer(
            claim_text,
            evidence_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "cid": cid
        }

        if "token_type_ids" in encoded:
            item["token_type_ids"] = encoded["token_type_ids"].squeeze(0)

        if self.has_label:
            item["labels"] = torch.tensor(
                LABEL2ID[c["claim_label"]],
                dtype=torch.long
            )

        return item

In [ ]:
train_ds_C4 = ClaimEvDatasetTransformer(
    train_claims,
    evidence,
    retrieval=None,          # train 用 GT evidence
    tokenizer=tokenizer_C4,
    max_len=256,
    max_evs=5
)

dev_ds_C4 = ClaimEvDatasetTransformer(
    dev_claims,
    evidence,
    retrieval=predict_R1,    # dev 用检索出来的 evidence
    tokenizer=tokenizer_C4,
    max_len=256,
    max_evs=5
)

train_loader_C4 = DataLoader(
    train_ds_C4,
    batch_size=8,
    shuffle=True
)

dev_loader_C4 = DataLoader(
    dev_ds_C4,
    batch_size=8,
    shuffle=False
)

print(
    f"#train = {len(train_ds_C4)}, "
    f"#dev = {len(dev_ds_C4)}"
)

#train = 1228, #dev = 154


In [ ]:
def train_transformer(model, train_loader, epochs=3, lr=2e-5):
    optimizer = AdamW(
        model.parameters(),
        lr=lr
    )

    for epoch in range(epochs):
        model.train()
        losses = []

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            inputs = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels
            }

            if "token_type_ids" in batch:
                inputs["token_type_ids"] = batch["token_type_ids"].to(device)

            outputs = model(**inputs)

            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

        print(f"Epoch {epoch+1}/{epochs} train_loss = {np.mean(losses):.4f}")

In [ ]:
train_transformer(
    model_C4,
    train_loader_C4,
    epochs=3,
    lr=2e-5
)

Epoch 1/3:   0%|          | 0/154 [00:00<?, ?it/s]

Epoch 1/3 train_loss = 1.1223


Epoch 2/3:   0%|          | 0/154 [00:00<?, ?it/s]

Epoch 2/3 train_loss = 0.8783


Epoch 3/3:   0%|          | 0/154 [00:00<?, ?it/s]

Epoch 3/3 train_loss = 0.6827


In [ ]:
def predict_with_transformer(model, data_loader):
    model.eval()

    pred_labels = {}

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            cids = batch["cid"]

            inputs = {
                "input_ids": input_ids,
                "attention_mask": attention_mask
            }

            if "token_type_ids" in batch:
                inputs["token_type_ids"] = batch["token_type_ids"].to(device)

            outputs = model(**inputs)

            preds = outputs.logits.argmax(dim=-1).cpu().tolist()

            for cid, pred_id in zip(cids, preds):
                pred_labels[cid] = ID2LABEL[pred_id]

    return pred_labels

In [ ]:
pred_label_C4 = predict_with_transformer(
    model_C4,
    dev_loader_C4
)

correct = sum(
    1
    for cid, pred in pred_label_C4.items()
    if dev_claims[cid]["claim_label"] == pred
)

acc_C4 = correct / len(pred_label_C4)

print(f"C4 BERT-base Classification Accuracy on dev: {acc_C4:.4f}")

Predicting:   0%|          | 0/20 [00:00<?, ?it/s]

C4 BERT-base Classification Accuracy on dev: 0.3961
